![ab_testing_image](ab_testing_image.jpg)

# A/B Testing: Evaluating a New Search Ranking Algorithm

### Business Objective

An online travel agency is considering introducing a new search ranking algorithm to improve booking conversion rates. The Product team wants to determine whether the new algorithm increases conversions without negatively affecting the booking experience.

The objective of this analysis is to evaluate the experiment results and provide a data-driven recommendation on whether to fully roll out the new algorithm.

### Data Overview

#### `sessions_data.csv`

| column | data type | description | 
|--------|-----------|-------------|
| `session_id` | `string` | Unique session identifier (unique for each row) |
| `user_id` | `string` | Unique user identifier (non logged-in users have missing user_id values; each user can have multiple sessions) |
| `session_start_timestamp` | `string` | When a session started |
| `booking_timestamp` | `string` | When a booking was made (missing if no booking was made during a session) |
| `time_to_booking` | `float` | time from start of the session to booking, in minutes (missing if no booking was made during a session) |
| `conversion` | `integer` | _New column to create:_ did session end up with a booking (0 if booking_timestamp or time_to_booking is Null, otherwise 1) |

<br>

#### `users_data.csv`

| column | data type | description | 
|--------|-----------|-------------|
| `user_id` | `string` | Unique user identifier (only logged-in users in this table) |
| `experiment_group` | `string` | control / variant split for the experiment (expected to be equal 50/50) |

<br>

## Methodology Overview

The experiment compares two groups:

- **Control:** Users exposed to the existing search ranking algorithm.
- **Variant:** Users exposed to the new search ranking algorithm.

### Metrics

| Metric | Type | Definition | Desired outcome |
|---|---|---|---|
| Conversion rate | Primary | Percentage of sessions resulting in a booking | Increase |
| Time to booking | Guardrail | Average time between session start and booking | Decrease or remain unchanged |

### Statistical Methods

| Analysis | Statistical method | Purpose |
|---|---|---|
| Sample allocation | Chi-square goodness-of-fit test | Assess whether observed session counts differ from the expected 50/50 allocation |
| Conversion rate | Two-proportion Z-test | Determine whether conversion rates differ between groups |
| Time to booking | Independent two-sample t-test | Determine whether average booking times differ between groups |
| Effect size | Relative change calculation | Quantify the magnitude and direction of the observed effects |

### Hypothesis Testing Framework

A 90% confidence level was selected, corresponding to a significance level of α = 0.10.

For both the primary and guardrail metrics, two-sided hypothesis tests were used:

- **Null hypothesis (H₀):** There is no difference between the control and variant groups.
- **Alternative hypothesis (Hₐ):** There is a difference between the control and variant groups.


### Rollout Decision Criteria

A full rollout is recommended only when both conditions are satisfied:

1. **Primary metric:** The conversion rate improvement is statistically significant (p-value below 0.10) and the effect is positive.
2. **Guardrail metric:** The difference in time to booking is either statistically insignificant (p-value above 0.10) or the effect is favorable (a decrease in booking time).

## Analysis

In [1]:
import pandas as pd
from scipy.stats import chisquare
from pingouin import ttest
from statsmodels.stats.proportion import proportions_ztest

In [2]:
sessions = pd.read_csv('sessions_data.csv')
users = pd.read_csv('users_data.csv')

In [3]:
sessions.sample(5)

,session_id,user_id,session_start_timestamp,booking_timestamp,time_to_booking
8703,0R7Ab2lgLymIrSqq,BFwSYUzsHxjX0Jxb,2025-01-21 17:49:13.817816734,NaN,NaN
765,sZhMQqsUOVbOi4My,RhB17IUrA0GQeExB,2025-01-24 00:13:56.786451340,NaN,NaN
14732,E40CL2s4bKizr4Qn,zoWsru5SKw2qubKu,2025-01-30 13:14:22.816379070,NaN,NaN
7244,f78Fv6igah3yM4AA,ww1NBROL2y18jIL2,2025-01-12 04:07:10.428704739,NaN,NaN
16028,jTIvAhGwHZwbIFWZ,NaN,2025-01-22 02:40:06.021267891,NaN,NaN


In [4]:
users.sample(5)

,user_id,experiment_group
8313,ixu22K6a3utFRNYc,control
7452,iZPuex44hPV5uHD1,control
5230,deSY69F7JPrmWpis,variant
2216,35zHoL4sMbsHBRWW,variant
5610,fMV02cKw8yztLY3Z,control


#### Step 1: Join dataframes

In [5]:
confidence_level = 0.90  # Set the pre-defined confidence level (90%)
alpha = 1 - confidence_level  # Significance level for hypothesis tests

In [6]:
sessions_x_users = pd.merge(sessions, users, on='user_id')
sessions_x_users.head(5)

,session_id,user_id,session_start_timestamp,booking_timestamp,time_to_booking,experiment_group
0,CP0lbAGnb5UNi3Ut,TcCIMrtQ75wHGXVj,2025-01-26 20:02:39.177358627,NaN,NaN,variant
1,UQAjrPYair63L1p8,TcCIMrtQ75wHGXVj,2025-01-20 16:12:51.536912203,NaN,NaN,variant
2,9zQrAPxV5oi2SzSa,TcCIMrtQ75wHGXVj,2025-01-28 03:46:40.839362144,NaN,NaN,variant
3,kkrz1M5vxrQ8wXRZ,GUGVzto9KGqeX3dc,2025-01-25 02:48:50.953303099,NaN,NaN,variant
4,ABZZFrwItZAPdYGP,v2EBIHmOdQfalI6k,2025-01-11 11:41:36.912253618,NaN,NaN,variant


#### Step 2: Calculate the conversion metric

In [7]:
sessions_x_users['conversion'] = sessions_x_users['booking_timestamp'].notna().astype(int)
sessions_x_users.head(5)

,session_id,user_id,session_start_timestamp,booking_timestamp,time_to_booking,experiment_group,conversion
0,CP0lbAGnb5UNi3Ut,TcCIMrtQ75wHGXVj,2025-01-26 20:02:39.177358627,NaN,NaN,variant,0
1,UQAjrPYair63L1p8,TcCIMrtQ75wHGXVj,2025-01-20 16:12:51.536912203,NaN,NaN,variant,0
2,9zQrAPxV5oi2SzSa,TcCIMrtQ75wHGXVj,2025-01-28 03:46:40.839362144,NaN,NaN,variant,0
3,kkrz1M5vxrQ8wXRZ,GUGVzto9KGqeX3dc,2025-01-25 02:48:50.953303099,NaN,NaN,variant,0
4,ABZZFrwItZAPdYGP,v2EBIHmOdQfalI6k,2025-01-11 11:41:36.912253618,NaN,NaN,variant,0


In [8]:
sessions_x_users.sample(5)

,session_id,user_id,session_start_timestamp,booking_timestamp,time_to_booking,experiment_group,conversion
8901,0HnFo4KK80IGVm5i,oIYK6Uo7RVHFlzN1,2025-01-22 14:23:02.196628332,NaN,NaN,control,0
9445,O40vcVLIR0ai5QSw,GW95atEN2WGckMtw,2025-01-07 20:41:15.377704144,NaN,NaN,variant,0
13579,rBjzQxSsvTNPc8AI,4hmr2o4fUzANEDKf,2025-01-12 22:28:50.293990850,NaN,NaN,variant,0
6202,rOgg9HLYaHi2xE0I,M7MQZH1rP7zHvWEi,2025-01-13 09:17:18.129343271,NaN,NaN,variant,0
3440,8H8Bb3Go8dL842Lt,2PoFft1XHYJfczQ1,2025-01-07 12:20:00.163545370,NaN,NaN,control,0


#### Step 3: Conduct a sanity check for Sample Ratio Mismatch between control and variant

In [36]:
# Check session-level allocation

# Get results in proportions
groups_count = sessions_x_users['experiment_group'].value_counts()
print(groups_count)

# Total sample size
n = sessions_x_users.shape[0]

# Examine the difference result statistically performing Chisquare test
chi = chisquare(f_obs = groups_count, f_exp=(n/2, n/2))
srm_chi2_pval = round(chi[1],4)
print('\nP-val:',srm_chi2_pval)

if srm_chi2_pval<alpha:
    print("SRM may be present")
else:
    print("SRM likely not present, and allocation matches the designed ratio")

experiment_group
variant    7653
control    7630
Name: count, dtype: int64

P-val: 0.8524
SRM likely not present, and allocation matches the designed ratio


**Interpretation:** Since p-value exceeds the 0.1 significance threshold, we fail to reject the null hypothesis of a 50/50 allocation. There is no statistically significant evidence of a sample ratio mismatch.

#### Step 4: Check the statistical significance of the experiment effect on primary metric (conversion)

In our experiment:
- Randomization happens at the user level: users are assigned to control or variant.
- Conversion is measured at the session level: each session either resulted in a booking (1) or did not (0).
- A user can have multiple sessions: the notebook explicitly says that user_id can appear in multiple rows of sessions_data.csv.

Therefore we will use session counts for the conversion test.

Significance test on conversion metric -> Choice: z-test
 - H0: Conversion rate is the same for exposed group and control group
 - Ha: Conversion rate isn't the same for exposed group and control group -->  Perform two-tailed test

Use *proportions_ztest()* where:
 - success counts = number of converted sessions per group
 - sample size = total number of sessions per group

In [18]:
# Calculate total number of sessions in each group
sample_sizes = sessions_x_users['experiment_group'].value_counts().loc[['control', 'variant']]

print(sample_sizes)

experiment_group
control    7630
variant    7653
Name: count, dtype: int64


In [19]:
# Calculate number of converted sessions in each group
success_counts = sessions_x_users.groupby('experiment_group')['conversion'].sum().loc[['control', 'variant']]

print(success_counts)

experiment_group
control    1215
variant    1392
Name: conversion, dtype: int64


In [20]:
# Perform proportions z-test
z_score, pval = proportions_ztest(
    count=success_counts,
    nobs=sample_sizes,
    alternative='two-sided'
)
pval_primary =  round(pval, 4)
print(pval_primary)

0.0002


#### Step 5: Check the statistical significance of the experiment effect on  guardrail metric (time_to_booking)

Significance test on time_to_booking metric -> Choice: t-test
- H0: Time to booking is same in both groups 
- Ha: time to booking is not the same in both groups --> Perform two-tailed test

In [23]:
# Check mean time_to_booking per group
sessions_x_users.groupby('experiment_group')['time_to_booking'].mean()

experiment_group
control    15.012404
variant    14.894029
Name: time_to_booking, dtype: float64

In [28]:
control_metric = sessions_x_users[sessions_x_users['experiment_group']=='control']['time_to_booking']
variant_metric = sessions_x_users[sessions_x_users['experiment_group']=='variant']['time_to_booking']

# Peform t-test
ttest_result = ttest(
    x=control_metric,
    y=variant_metric,
    alternative='two-sided'
)

pval_guardrail = ttest_result['p_val'].values[0]
pval_guardrail = round(pval_guardrail,4)
print('P-val:',pval_guardrail)

P-val: 0.5365


#### # Step 6. Estimate the effect sizes on the primary and guardrail metrics

In [32]:
avg_control_primary = sessions_x_users[sessions_x_users['experiment_group'] == 'control']['conversion'].mean()
avg_variant_primary = sessions_x_users[sessions_x_users['experiment_group'] == 'variant']['conversion'].mean()

# Estimate effect size for the conversion metric
effect_size_primary = avg_variant_primary/avg_control_primary - 1
effect_size_primary = round(effect_size_primary, 4)
print(f'\nPrimary metric effect size: {effect_size_primary: .4f}')


Primary metric effect size:  0.1422


In [33]:
avg_control_guardrail = sessions_x_users[sessions_x_users['experiment_group'] == 'control']['time_to_booking'].mean()
avg_variant_guardrail = sessions_x_users[sessions_x_users['experiment_group'] == 'variant']['time_to_booking'].mean()

# Calculate effect size for guardrail metric
effect_size_guardrail = avg_variant_guardrail/avg_control_guardrail - 1
effect_size_guardrail = round(effect_size_guardrail, 4)
print(f'\nGuardrail effect size: {effect_size_guardrail}')


Guardrail effect size: -0.0079


#### # Step 7: Making a decision 

In [34]:
# Primary metric must be statistically significant and show positive effect (increase)
criteria_full_on_primary = (pval_primary < alpha) & (effect_size_primary > 0)

# Guardrail must either be statistically insignificant or show positive effect (decrease)
criteria_full_on_guardrail = (pval_guardrail > alpha) | (effect_size_guardrail <= 0)

# Final launch decision based on both metrics
if criteria_full_on_primary and criteria_full_on_guardrail:
    decision_full_on = 'Yes'
    print('\nThe experiment results are significantly positive and the guardrail metric was not harmed, we are going full on!')
else:
    decision_full_on = 'No'
    print('\nThe experiment results are inconclusive or the guardrail metric was harmed, we are pulling back!')


The experiment results are significantly positive and the guardrail metric was not harmed, we are going full on!


### Key Findings Summary

| Metric | Control | Variant | Result |
|---|---:|---:|---|
| Session conversion rate | 15.92% | 18.19% | +14.22% relative change |
| Average time to booking | 15.01 min | 14.89 min | -0.79% relative change |

**Primary metric — Conversion**

The variant achieved a 14.22% relative increase in session conversion rate. The difference was statistically significant (p = 0.0002), providing evidence that the new algorithm improved conversion performance.

**Guardrail metric — Time to booking**

Average time to booking decreased by 0.79% in the variant group. However, the difference was not statistically significant (p = 0.5365). There is no statistically significant evidence that the new algorithm increased the time required to book.

**Sample allocation check**

The session-level allocation check returned a p-value of 0.8524. We fail to reject the null hypothesis of an equal 50/50 session allocation.

**Recommendation:** Proceed with a full rollout of the new search ranking algorithm, based on the results and decision criteria used in this analysis.

### Limitations and Considerations

- **Repeated sessions:** Users could generate multiple sessions, meaning observations may not be fully independent. The session-level statistical tests do not account for potential correlation between sessions from the same user.
- **Guardrail metric:** Time to booking was evaluated only among sessions that resulted in a booking, so the analysis does not capture the experience of users who did not book.
- **Scope of the allocation check:** The allocation check compares session counts between groups, while experiment assignment was performed at the user level.

These limitations should be considered when interpreting the results and planning future experiments.